# Magnetogram : Carrington map to full solar disk conversion

In [ ]:
import pandas as pd
import os
from datetime import datetime

# List to collect rows
rows = []

for year in range(1975, 2004):   # inclusive 1975 → 2003
    for month in range(1, 13):   # full 12 months
        image_path = f"//home/iron-man/Documents/long_term_reconstruction/callibrated/Cak_Rotated/{year}/{month:02d}"
        if os.path.exists(image_path): 
            contents = os.listdir(image_path)
            for content in contents:
                file_name = content.split(".")[0]     # e.g. 19750113_0218
                if "_" not in file_name:
                    continue  # skip unexpected files
                date = file_name.split('_')[0]        # 19750113
                time = file_name.split('_')[1]        # 0218

                # build datetime safely
                try:
                    dt = datetime.strptime(date + time, "%Y%m%d%H%M")
                except ValueError:
                    continue  # skip malformed filenames

                rows.append({
                    "year": dt.year,
                    "month": dt.month,
                    "day": dt.day,
                    "hour": dt.hour,
                    "minute": dt.minute,
                    "datetime": dt,
                    "value": file_name
                })

# Convert to DataFrame
df = pd.DataFrame(rows)

# Save to CSV
df.to_csv("cak_data_paths.csv", index=False)

print("Saved cak_data_paths.csv with", len(df), "rows")
df.head()


In [ ]:
import os

mag_path = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/solisdata/Syn_Maps_Br_fromlos_OLD"
Carrington_no = []

if os.path.exists(mag_path): 
    crr_series = os.listdir(mag_path)
    for filename in crr_series:                # e.g. "m1648f.fits"
        if filename.startswith("m") and filename.endswith(".fits"):
            Number = filename.split("m")[1]    # → "1648f.fits"
            Carrington_number = Number.split("f")[0]   # → "1648"
            
            if Carrington_number.isdigit():
                Carrington_number = int(Carrington_number)
                Carrington_no.append(Carrington_number)


In [ ]:
df1 = pd.read_csv("/home/iron-man/Documents/long_term_reconstruction/carrington_rotation_start_dates.csv")  
df1 = df1[df1["Rotation"].isin(Carrington_no)]
print(df1)


In [ ]:
df1

In [ ]:
import numpy as np 
print(np.sort(Carrington_no)[300:320])

In [ ]:
from astropy.time import Time
from sunpy.coordinates import sun
from astropy.io import fits
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from datetime import datetime, timedelta
from astropy.io import fits

nlat = 180
nlon = 360
npix = 256
disk_radius=80
disk_radius_px=npix*0.898/2
for value in np.sort(Carrington_no):
    mag_path_center = f"/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/solisdata/Syn_Maps_Br_fromlos_OLD/m{value}f.fits"
    mag_path_right = f"/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/solisdata/Syn_Maps_Br_fromlos_OLD/m{value-1}f.fits"
    mag_path_left = f"/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/solisdata/Syn_Maps_Br_fromlos_OLD/m{value+1}f.fits"

    # load magnetogram Carrington maps 
    Br_center = fits.getdata(mag_path_center)



    
    # get times
    row_1 = df1[df1["Rotation"] == value]
    row_2 = df1[df1["Rotation"] == (value + 1)]

    if row_1.empty or row_2.empty:
        continue
    row_1 = row_1.iloc[0]
    row_2 = row_2.iloc[0]
    
    year, month = int(row_1["Year"]), int(row_1["Month"])
    day_fraction = float(row_1["Day"])
    day, frac = int(day_fraction), day_fraction - int(day_fraction)
    t_start = datetime(year, month, day) + timedelta(days=frac)

    year, month = int(row_2["Year"]), int(row_2["Month"])
    day_fraction = float(row_2["Day"])
    day, frac = int(day_fraction), day_fraction - int(day_fraction)
    t_end = datetime(year, month, day) + timedelta(days=frac)

    # filter df between t_start and t_end
    mask = (df["datetime"] >= t_start) & (df["datetime"] < t_end)
    df_filtered = df[mask]




    
    for time_value in df_filtered["datetime"]:
        t = Time(time_value)
        L0 = sun.L0(t).to_value("deg")
        B0 = sun.B0(t).to_value("deg")
        

        # ---- Decide whether to merge maps ----
        if L0 + disk_radius > 360:
            if os.path.isfile(mag_path_right):
                Br_right = fits.getdata(mag_path_right)
                # near the right edge → merge center + right
                Br = np.hstack((Br_center, Br_right))
                Br = Br[:,int(L0)-180:int(L0)+180]
                L0=180
            else:
                continue

        elif L0 - disk_radius < 0:
            if os.path.isfile(mag_path_left):
                Br_left = fits.getdata(mag_path_left)
                # near the left edge → merge left + center
                Br = np.hstack((Br_left, Br_center))
                L0=L0+360     # shift L0 by +360 because the left map is now in [-360,0]
                Br = Br[:,int(L0)-180:int(L0)+180]
                L0=180 # now L0 become center
            else:
                continue

        else:
            # use only one Carrington map → 360 longitude grid
            Br = Br_center

        # build Lon/Lat grid matching Br shape
        nlat, nlon_actual = Br.shape
        lon = np.linspace(0, 359, nlon)
        sinlat = np.linspace(-1, 1, nlat)                        # uniform in sin(latitude)
        lat = np.degrees(np.arcsin(sinlat))                      # convert to actual latitude
        Lon, Lat = np.meshgrid(lon, lat)

        

        Lon_rad = np.deg2rad(Lon - L0)
        Lat_rad = np.deg2rad(Lat)
        B0_rad  = np.deg2rad(B0)

                
        r=1
        x = r*(np.cos(Lat_rad)*np.sin(Lon_rad))     # x = r cos Θ sin(Φ − Φ0),
        y = r*(np.sin(Lat_rad)*np.cos(B0_rad) - np.cos(Lat_rad)*np.cos(Lon_rad)*np.sin(B0_rad))      # y = r[sin Θ cos B0 − cos Θ cos(Φ − Φ0)sin B0] , 
        z = r*(np.sin(Lat_rad)*np.sin(B0_rad) + np.cos(Lat_rad)*np.cos(Lon_rad)*np.cos(B0_rad))      # z = r[sin Θ sin B0 + cos Θ cos(Φ − Φ0) cos B0]
        
        visible = z > 0
        X = x[visible]
        Y = y[visible]
        values = Br[visible]

        # scale to pixel coords
        X_pix = (X * disk_radius_px + npix/2).ravel()
        Y_pix = (Y * disk_radius_px + npix/2).ravel()

        # make pixel grid
        grid_x, grid_y = np.meshgrid(np.arange(npix), np.arange(npix))
        
        # interpolate onto image plane
        Br_cropped = griddata(
            points=(X_pix, Y_pix),
            values=values.ravel(),
            xi=(grid_x, grid_y),
            method="linear",
            fill_value= np.nan
        )
        
        R = np.sqrt((grid_x - npix/2)**2 + (grid_y - npix/2)**2)
        Br_cropped[R > disk_radius_px] = np.nan
        plt.imshow(Br_cropped)
        plt.show()
        save_dir = f"/home/iron-man/Documents/long_term_reconstruction/callibrated/Cropped_mag_new_1/{time_value.year}/{time_value.month:02d}"
        # Ensure output directory exists
        os.makedirs(save_dir, exist_ok=True)
        file_name = f"{time_value.strftime('%Y%m%d_%H%M')}.fits"
        output_path = os.path.join(save_dir, file_name)
        # finding the missing values in magnetogram which is band of zeros
        missing_value = (Br_cropped==0).sum()
        
        if missing_value<=2000:
            # Save Br_cropped as FITS
            hdu = fits.PrimaryHDU(Br_cropped)       # 19070208_0827.h5
            hdul = fits.HDUList([hdu])

            # Optionally add metadata to header
            hdr = hdul[0].header
            hdr['DATE'] = time_value.strftime("%Y-%m-%d %H:%M:%S")
            hdr['L0'] = L0
            hdr['B0'] = B0
            hdr['DISKRAD'] = disk_radius
            hdr['CARR_ROT'] = value

            # Write file (overwrite if exists)
            hdul.writeto(output_path, overwrite=True)
            print(f"completed{file_name}")
print(f"completed")    


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import resize

# File path
image_path = "/home/rohan/Documents/data/callibrated/Cak_Rotated/1989/12/19891231_0235.h5"

# Open the file in read mode
with h5py.File(image_path, "r") as f:
    print("Keys:", list(f.keys()))
    data = f["data"][:]  # HDF5 dataset
    print(data.shape, data.dtype)
    

# Resize to 256x256
image = resize(data, (256, 256), order=1, preserve_range=True).astype(np.float32)
image = np.clip(image, 0, 1)

# --- Thresholding ---
# Adjust thresholds depending on brightness of your dataset
lower_thresh, upper_thresh = 0.2, 1.0
binary_mask = np.where((image >= lower_thresh) & (image <= upper_thresh), 1, 0)

# --- Plot original + thresholded ---
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

axs[0].imshow(image, cmap='gray', origin='lower')
axs[0].set_title("Original CaK")

axs[1].imshow(binary_mask, cmap='gray', origin='lower')
axs[1].set_title(f"Thresholded ({lower_thresh}-{upper_thresh})")

plt.show()

# plt.contour(binary_mask, colors='blue', linewidths=0.8)


In [ ]:
import sunpy.map
from sunpy.coordinates import Helioprojective, HeliographicCarrington
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits
# from reproject import reproject_interp


mag_path = "/home/rohan/Documents/data/callibrated/Cropped_mag/1989/12/19891231_0235.fits"
# --- Process Magnetogram ---
with fits.open(mag_path) as hdul:
    mag_img = hdul[0].data
    header = hdul[0].header

plt.imshow(mag_img,origin='lower',cmap='gray')
print(data.shape, data.dtype)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 4))

# Show the target image as the background
plt.imshow(mag_img,origin='lower', cmap='gray')

# Overlay the contour from the input image
contour = plt.contour(binary_mask, colors='blue', linewidths=0.2)

plt.title("Input Image Contour over Target Image")
plt.axis('off')  # Optional: hide axis
plt.tight_layout()
plt.show()
